[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensioai/blog/blob/master/028_pytorch_nn/pytorch_nn.ipynb)

In [79]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Pytorch - Redes Neuronales

En el post [anterior](https://sensioai.com/blog/027_pytorch_intro) hicimos una introducción al framework de `redes neuronales` `Pytorch`. Hablamos de sus tres elementos fundamentales: el objeto `tensor` (similar al `array` de `NumPy`) `autograd` (que nos permite calcular derivadas de manera automáticas) y el soporte GPU. En este post vamos a entrar en detalle en la  funcionalidad que nos ofrece la librería para diseñar redes neuronales de manera flexible.

In [80]:
import torch
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score

# CPU (no CUDA)
device = torch.device("cpu")

In [81]:
# Cargar dataset
data = pd.read_csv('/content/drive/MyDrive/IA/DATASETS/Electric_Vehicle_Population_Data (1).csv')

# Revisar nombres de columnas
print(data.columns)

Index(['VIN (1-10)', 'County', 'City', 'State', 'Postal Code', 'Model Year',
       'Make', 'Model', 'Electric Vehicle Type',
       'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range',
       'Base MSRP', 'Legislative District', 'DOL Vehicle ID',
       'Vehicle Location', 'Electric Utility', '2020 Census Tract'],
      dtype='object')


## Modelos secuenciales

La forma más sencilla de definir una `red neuronal` en `Pytorch` es utilizando la clase `Sequentail`. Esta clase nos permite definir una secuencia de capas, que se aplicarán de manera secuencial (las salidas de una capa serán la entrada de la siguiente). Ésto ya lo conocemos de posts anteriores, ya que es la forma ideal de definir un `Perceptrón Multicapa`.

In [82]:
# ===== Versión mínima de preproceso =====
TARGET_COL = "Electric Vehicle Type"  # cambia si quieres otra etiqueta
df = data.dropna(subset=[TARGET_COL]).copy()

# Evita columnas gigantes (opcional pero recomendado)
df = df.drop(columns=[c for c in ["VIN (1-10)", "DOL Vehicle ID", "Vehicle Location", "2020 Census Tract", "Postal Code"] if c in df.columns])

import pandas as pd, numpy as np
X = pd.get_dummies(df.drop(columns=[TARGET_COL]), dummy_na=True).astype(np.float32)

from sklearn.preprocessing import LabelEncoder
lbl = LabelEncoder()
y = lbl.fit_transform(df[TARGET_COL].astype(str)).astype(np.int64)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.15, random_state=42, stratify=y
)


In [83]:
num_classes = len(np.unique(y_train))
D_in, H, D_out = X_train.shape[1], 100, num_classes

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

# D_in, H1, H2, D_out = 784, 100, 50, 10
# model = torch.nn.Sequential(
#     torch.nn.Linear(D_in, H1),
#     torch.nn.ReLU(),
#     torch.nn.Linear(H1, H2),
#     torch.nn.ReLU(),
#     torch.nn.Linear(H2, D_out),
# )


El modelo anterior es un `MLP` con 784 entradas, 100 neuronas en la capa oculta y 10 salidas. Podemos usar este modelo para hacer un clasificador de imágenes con el dataset MNIST. Pero primero, vamos a ver como podemos calcular las salidas del modelo a partir de unas entradas de ejemplo.

In [84]:
x_prueba = torch.randn(600, D_in, device=device)
outputs = model(x_prueba)
print(outputs.shape)        # (600, D_out = num_classes)

torch.Size([600, 2])


In [85]:
print(outputs[0][:])

tensor([-0.0675,  0.0882], grad_fn=<SliceBackward0>)


Como puedes ver, simplemente le pasamos los inputs al modelo (llamándolo como una función). En este caso, usamos un tensor con 64 vectores de 784 valores. Es importante remarcar que los modelos de `Pytorch` (por lo general) siempre esperan que la primera dimensión sea la dimensión *batch*. Si queremos entrenar esta red en una GPU, es tan sencillo como

In [86]:
model

Sequential(
  (0): Linear(in_features=1421, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=2, bias=True)
)

In [87]:
model.to("cuda")

Sequential(
  (0): Linear(in_features=1421, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=2, bias=True)
)

Vamos a ver ahora como entrenar este modelo con el dataset MNIST.

In [88]:
device = torch.device("cpu")
model = model.to(device)

X_t = torch.from_numpy(X_train).float().to(device)
Y_t = torch.from_numpy(y_train).long().to(device)

# Chequeo de seguridad (evita el error de shapes)
assert model[0].in_features == X_t.shape[1], f"{model[0].in_features} vs {X_t.shape[1]}"


In [89]:
# función de pérdida y derivada

def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1, keepdims=True)

def cross_entropy(logits, targets):
    return (torch.logsumexp(logits, dim=1) - logits[torch.arange(logits.size(0)), targets]).mean()
    loss = cross_entropy(y_pred, Y_t)
    return loss

In [90]:
# X_train

In [91]:
#torch.cuda.is_available()

In [92]:
print(X)

        Model Year  Electric Range  Base MSRP  Legislative District  \
0           2019.0           220.0        0.0                  15.0   
1           2024.0            21.0        0.0                  35.0   
2           2022.0            26.0        0.0                  32.0   
3           2017.0            14.0        0.0                  30.0   
4           2013.0            75.0        0.0                  40.0   
...            ...             ...        ...                   ...   
257630      2020.0            32.0        0.0                  21.0   
257631      2022.0             0.0        0.0                  48.0   
257632      2019.0            15.0    55700.0                  18.0   
257633      2019.0            25.0        0.0                  40.0   
257634      2025.0             0.0        0.0                  43.0   

        County_Ada  County_Adams  County_Alameda  County_Albemarle  \
0              0.0           0.0             0.0               0.0   
1      

In [93]:
# Forzar TODO a CPU
device = torch.device("cpu")

# 1) Modelo en CPU
model = model.to(device)   # o: model = model.cpu()

# 2) Datos en CPU
X_t = torch.from_numpy(X_train).float()
Y_t = torch.from_numpy(y_train).long()

# (opcional) chequeo rápido
print("model device:", next(model.parameters()).device)
print("X_t device:", X_t.device, "| Y_t device:", Y_t.device)


model device: cpu
X_t device: cpu | Y_t device: cpu


In [94]:
# convertimos datos a tensores y copiamos en gpu

#X_t = torch.from_numpy(X_train).float()
#Y_t = torch.from_numpy(y_train).long()

# bucle entrenamiento
epochs = 50
lr = 0.2
log_each = 10
for e in range(1, epochs + 1):
    model.train()
    y_pred = model(X_t)
    loss = cross_entropy(y_pred, Y_t)

    model.zero_grad()
    loss.backward()
    with torch.no_grad():
        for p in model.parameters():
            p -= lr * p.grad

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {loss.item():.5f}")

Epoch 10/50 Loss nan
Epoch 20/50 Loss nan
Epoch 30/50 Loss nan
Epoch 40/50 Loss nan
Epoch 50/50 Loss nan


Como puedes observar en el ejemplo, podemos calcular la salida del modelo con una simple línea. Luego calculamos la función de pérdida, y llamando a la función `backward` `Pytorch` se encarga de calcular las derivadas de la misma con respecto a todos los parámetros del modelo automáticamente (si no queremos acumular estos gradientes, nos aseguramos de llamar a la función `zero_grad` para ponerlos a cero antes de calcularlos). Por útlimo, podemos iterar por los parámetros del modelo aplicando la regla de actualización deseada (en este caso usamos `descenso por gradiente`).

In [100]:
from sklearn.metrics import accuracy_score

def evaluate(x):
    model.eval()
    with torch.no_grad():
        y_pred = model(x)
        y_probas = softmax(y_pred)
        return torch.argmax(y_probas, dim=1)

y_pred = evaluate(torch.from_numpy(X_test).float().to(device))
acc = accuracy_score(y_test, y_pred.cpu().numpy())
print("Accuracy:", acc)

Accuracy: 0.796072038503338


Existen algunos tipos de capas que se comportan diferente en función de si estamos entrenando la red o usándola para generar predicciones. Podemos controlar el modo en el que queremos que esté nuestra red con las funciones `train` y `eval`.

## Optimizadores y Funciones de pérdida

En el ejemplo anterior hemos calculado la función de pérdida y aplicado la regla de optimización de forma manual. Sin embargo, `Pytorch` nos ofrece funcionalidad que nos abstrae estos cálculos ofreciendo además flexibilidad para aplicar diferentes funciones de pérdida o algoritmos de optimización de manera sencilla. Podemos encontrar diferentes funciones de pérdida ya implementadas en el paquete `torch.nn`.

In [101]:
criterion = torch.nn.CrossEntropyLoss()

Mientras que los optimizadores se encuentran en el paquete `torch.optim`

In [102]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

Puedes ver la lista completa de funciones de pérdida y optimizadores disponibles en la [documentación](https://pytorch.org/docs/stable/index.html), aunque como ya has visto siempre puedes definir los tuyos propios fácilmente.

Una vez definidos estos dos objetos, nuestro bucle de entrenamiento se simplifica considerablemente.

In [112]:
D_in, H, D_out = X_train.shape[1], 100, num_classes

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

epochs = 50
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float())
print(accuracy_score(y_test, y_pred.cpu().numpy()))

Epoch 10/50 Loss nan
Epoch 20/50 Loss nan
Epoch 30/50 Loss nan
Epoch 40/50 Loss nan
Epoch 50/50 Loss nan
0.796072038503338


## Modelos custom

Si bien en muchos casos definir una `red neuronal` como una secuencia de capas es suficiente, en otros casos será un factor limitante. Un ejemplo son las redes residuales, en las que no sólo utilizamos la salida de una capa para alimentar la siguiente si no que, además, le sumamos su propia entrada. Este tipo de arquitectura no puede ser definida con la clase `Sequential`, y para ello necesitamos hacer un modelo *customizado*. Para ello, `Pytroch` nos ofrece la siguiente sintaxis.

In [109]:
# creamos una clase que hereda de `torch.nn.Module`

class ModeloPersonalizado(torch.nn.Module):

    # constructor
    def __init__(self, D_in, H, D_out):

        # llamamos al constructor de la clase madre
        super(ModeloPersonalizado, self).__init__()

        # definimos nuestras capas
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    # lógica para calcular las salidas de la red
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

En primer lugar, necesitamos definir una nueva clase que herede de la clase `torch.nn.Module`. Esta clase madre aportará toda la funcionalidad esencial que necesita una `red neuronal` (soporte GPU, iterar por sus parámeteros, etc). Luego, en esta clase necesitamos definir mínimos dos funciones:

- `init`: en el constructor llamaremos al constructor de la clase madre y después definiremos todas las capas que querramos usar en la red.
- `forward`: en esta función definimos toda la lógica que aplicaremos desde que recibimos los inputs hasta que devolvemos los outputs.

En el ejemplo anterior simplemente hemos replicado la misma red (puedes conseguir el mismo efecto usando la clase `Sequential`).

In [110]:
model = ModeloPersonalizado(784, 100, 10)
# Codigo para saber si el modelo esta votando los datos en las cantidades correctas
x_prueba = torch.randn(500, 784)
print(x_prueba)
outputs = model(x_prueba)
outputs.shape

tensor([[-0.4531, -0.5898,  0.7928,  ...,  0.8991, -1.2727, -0.5756],
        [ 1.1421, -0.1104,  0.0894,  ...,  0.7403,  0.4923,  0.6053],
        [-0.1056,  0.2701, -0.4074,  ...,  0.0821,  1.2401,  0.7577],
        ...,
        [ 0.1007, -0.1763,  0.3365,  ..., -0.1595,  0.5797,  1.3088],
        [-0.5791, -0.5419, -0.6773,  ..., -0.1849, -1.1372, -0.1824],
        [ 1.5497, -1.1801,  1.5042,  ..., -0.7709,  0.9394, -0.0669]])


torch.Size([500, 10])

Ahora, podemos entrenar nuestra red de la misma forma que lo hemos hecho anteriormente.

In [111]:
model.to("cuda")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cuda())
accuracy_score(y_test, y_pred.cpu().numpy())

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_addmm)

Aquí puedes ver otro ejemplo de como definir un `MLP` con conexiones residuales, algo que no podemos hacer simplemente usando un modelo secuencial.

In [113]:
class ModelCustom2(torch.nn.Module):

    def __init__(self, D_in, H, D_out):
        super(ModelCustom2, self).__init__()
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    def forward(self, x):
        x1 = self.fc1(x)
        x = self.relu(x1)
        x = self.fc2(x + x1)
        return x

In [114]:
model = ModelCustom2(784, 100, 10).to("cuda")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cuda())
accuracy_score(y_test, y_pred.cpu().numpy())

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_addmm)

De esta manera, tenemos mucha flexibilidad para definir nuestras redes.

## Accediendo a las capas de una red

En ocasiones queremos acceder a una capa en particular de nuestra red. Para ello, podemos acceder utilizando su nombre.

In [ ]:
model

In [ ]:
model.fc1

También podemos acceder directamente a los tensores que contienen los parámetros con las propiedades adecuadas

In [ ]:
model.fc1.weight

In [ ]:
model.fc1.bias

Es posible sobreescribir una capa de la siguiente manera

In [ ]:
model.fc2 = torch.nn.Linear(100, 1)

model

Ahora, la capa final de nuestra red tiene solo una salida. Esta nueva capa ha sido inicializada de manera aleatoria, por lo que esta nueva red no nos va a servir de mucho. Sin embargo, podríamos volver a entrenar esta red en otro problema en el que solo necesitemos una salida aprovechando los pesos que ya hemos entrenado anteriormente con el dataset MNIST. Esto es la base del *transfer learning*, una técnica que utilizaremos muchísimo más adelante y la cual explicaremos en detalle.

A continuación encontrarás varios trucos a la hora de crear redes neuronales a partir de otras que te pueden resultar útiles.

In [ ]:
# obtener una lista con las capas de una red

list(model.children())

In [ ]:
# crear nueva red a partir de la lista (excluyendo las útlimas dos capa)

new_model = torch.nn.Sequential(*list(model.children())[:-2])
new_model

In [ ]:
# crear nueva red a partir de la lista (excluyendo las útlima capa)

new_model = torch.nn.ModuleList(list(model.children())[:-1])
new_model

## Resumen

En este post hemos visto la funcionalidad que `Pytorch` nos ofrece a la hora de definir y entrenar nuestras `redes neuronales`. El paquete `torch.nn` contiene todo lo necesario para diseñar nuestros modelos, ya sea de manera secuencial o con una clase *custom* para arquitecturas más complicadas. También nos da muchas funciones de pérdida que podemos usar directamente para entrenar las redes. Te recomiendo encarecidamente que le eches un vistazo a la [documentación](https://pytorch.org/docs/stable/nn.html) par hacerte una idea de todo lo que puedes hacer. También hemos visto como el paquete `torch.optim` nos oferece algoritmos de optimización que también nos hacen la vida más fácil a la hora de entrenar nuestras redes.